# 12 — Integrated Results
## Parkinson metabolic transcriptomic signature

Final integration of Notebooks 03–11 without retraining models. Evidence layers: bulk DE → metabolic prioritization → snRNA-seq validation → cross-dataset validation → mechanistic network → exploratory ML → XAI.

**Interpretation:** scores are prioritization scores, not probabilities; transcript abundance is not enzyme activity or metabolic flux; ML/XAI with n=10 is exploratory/apparent rather than clinical validation.


In [ ]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
ROOT=Path(r"C:\Users\Camfe\pythonlab")
OUT=ROOT/"results"/"integrated_results"/"notebook12"
FIG=ROOT/"figures"/"integrated_results"/"notebook12"
OUT.mkdir(parents=True,exist_ok=True); FIG.mkdir(parents=True,exist_ok=True)
print("ROOT:",ROOT); print("OUT:",OUT)


## 1. Locate previous outputs


In [ ]:
def locate(name, preferred=None, required=False):
    if preferred is not None and Path(preferred).exists(): return Path(preferred)
    hits=list(ROOT.rglob(name))
    if hits: return hits[0]
    if required: raise FileNotFoundError(f"Required file not found: {name}")
    return None

F={
"primary":locate("GSE136666_SN_primary_metabolic_signature.csv",ROOT/"data"/"processed"/"notebook05"/"GSE136666_SN_primary_metabolic_signature.csv",True),
"priority":locate("GSE136666_SN_metabolic_candidates_prioritized.csv",ROOT/"data"/"processed"/"notebook05"/"GSE136666_SN_metabolic_candidates_prioritized.csv",True),
"sn_validation":locate("GSE184950_Notebook05_primary_signature_validation.csv"),
"cross_class":locate("cross_dataset_gene_final_classification.csv",None,True),
"cross_evidence":locate("cross_dataset_evidence_table.csv"),
"mechanistic":locate("mechanistic_gene_priority.csv",None,True),
"mechanistic_final":locate("final_gene_protein_enzyme_reaction_table.csv"),
"ml_metrics":locate("LOOCV_metrics_all_models.csv",None,True),
"ml_predictions":locate("primary_LOOCV_predictions.csv",None,True),
"ml_permutations":locate("primary_LOOCV_AUC_permutations.csv"),
"xai":locate("thesis_XAI_interpretation_table.csv",None,True),
"xai_samples":locate("sample_level_XAI_summary.csv")}
for k,v in F.items(): print(f"{k:18s}: {v}")
print("\n✓ Required integration inputs found.")


## 2. Load and audit tables


In [ ]:
T={}
for k,p in F.items():
    if p is not None:
        T[k]=pd.read_csv(p)
        print(k,T[k].shape,T[k].columns.tolist())


## 3. Harmonize gene identifiers


In [ ]:
def detect(df,names=(),contains=(),required=False,label="column"):
    low={str(c).lower():c for c in df.columns}
    for n in names:
        if n.lower() in low:return low[n.lower()]
    for c in df.columns:
        cl=str(c).lower()
        if contains and all(x.lower() in cl for x in contains):return c
    if required:raise KeyError(f"Could not detect {label}. Columns: {df.columns.tolist()}")
    return None

def gene_table(df):
    d=df.copy(); gc=detect(d,["signature_gene","gene","Symbol","symbol","Gene"],required=True,label="gene")
    d["gene"]=d[gc].astype(str).str.strip(); return d

G={}
for k in ["primary","priority","sn_validation","cross_class","cross_evidence","mechanistic","mechanistic_final","xai"]:
    if k in T:
        try:G[k]=gene_table(T[k])
        except Exception as e:print("Skipped",k,e)
genes=G["primary"]["gene"].dropna().drop_duplicates().tolist()
if len(genes)<3:raise RuntimeError("Primary signature unexpectedly small.")
print("Primary signature:",len(genes),genes)


## 4. Master gene evidence table


In [ ]:
master=pd.DataFrame({"gene":genes})
def merge_cols(master,key,cols):
    if key not in G:return master
    d=G[key].drop_duplicates("gene")
    keep=["gene"]+[c for c in cols if c in d.columns and c!="gene" and c not in master.columns]
    return master.merge(d[keep],on="gene",how="left") if len(keep)>1 else master

master=merge_cols(master,"priority",["priority_score","evidence_tier","expected_bulk_direction","log2FoldChange","padj","metabolic_pathway_count","metabolic_theme_count","metabolic_themes","pathway_database_count"])
master=merge_cols(master,"cross_class",["cross_dataset_score","final_validation_class","bulk_log2FC","bulk_padj","median_sn_log2FC","concordance_fraction","n_sn_FDR","n_both_FDR"])
master=merge_cols(master,"cross_evidence",["cross_dataset_score","final_validation_class","bulk_log2FC","bulk_padj","median_sn_log2FC","concordance_fraction","n_sn_FDR","n_both_FDR"])
master=merge_cols(master,"mechanistic",["mechanistic_priority_score","mechanistic_rank","degree","betweenness","theme_count","enzyme_role","protein","reaction","metabolic_theme"])
master=merge_cols(master,"xai",["standardized_beta","standardized_logistic_coefficient","absolute_beta","absolute_beta_rank","mean_absolute_sample_contribution","dominant_sign_fraction","sign_stable_all_folds","XAI_interpretation"])
if "standardized_beta" not in master and "standardized_logistic_coefficient" in master: master["standardized_beta"]=master["standardized_logistic_coefficient"]
display(master); master.to_csv(OUT/"integrated_master_gene_evidence.csv",index=False)


## 5. Evidence coverage and integrated categories


In [ ]:
rows=[]
for label,col in {"Discovery":"priority_score","Cross-dataset":"final_validation_class","Mechanistic":"mechanistic_priority_score","Primary ML/XAI":"standardized_beta"}.items():
    if col in master:
        n=int(master[col].notna().sum()); rows.append({"layer":label,"genes_with_evidence":n,"signature_genes":len(master),"coverage":n/len(master)})
coverage=pd.DataFrame(rows); display(coverage); coverage.to_csv(OUT/"integrated_evidence_coverage.csv",index=False)
if "final_validation_class" not in master:master["final_validation_class"]="Not_available"
def category(r):
    base=str(r.get("final_validation_class","Not_available")); ann=[]
    if pd.notna(r.get("mechanistic_priority_score",np.nan)):ann.append("mechanistic")
    if pd.notna(r.get("standardized_beta",np.nan)):ann.append("primary_ML_XAI")
    return base+(" | "+",".join(ann) if ann else "")
master["integrated_evidence_category"]=master.apply(category,axis=1)
master.to_csv(OUT/"integrated_master_gene_evidence.csv",index=False)


## 6. Discussion order — preserving existing evidence hierarchy


In [ ]:
ranked=master.copy()
ranked["_m"]=pd.to_numeric(ranked["mechanistic_rank"],errors="coerce") if "mechanistic_rank" in ranked else np.nan
ranked["_c"]=pd.to_numeric(ranked["cross_dataset_score"],errors="coerce") if "cross_dataset_score" in ranked else np.nan
ranked=ranked.sort_values(["_m","_c"],ascending=[True,False],na_position="last").drop(columns=["_m","_c"])
ranked["discussion_order"]=range(1,len(ranked)+1)
ranked.to_csv(OUT/"integrated_gene_discussion_order.csv",index=False); display(ranked)


## 7. Validation summary and figure


In [ ]:
vc=master["final_validation_class"].fillna("Not_available").value_counts().rename_axis("validation_class").reset_index(name="n_genes")
display(vc); vc.to_csv(OUT/"validation_class_summary.csv",index=False)
fig,ax=plt.subplots(figsize=(8,4.8)); ax.bar(vc.validation_class,vc.n_genes); ax.set_ylabel("Genes"); ax.set_title("Cross-dataset validation classes"); ax.tick_params(axis="x",rotation=35)
fig.tight_layout(); fig.savefig(FIG/"validation_classes.png",dpi=300,bbox_inches="tight"); plt.show()


## 8. Bulk versus snRNA-seq direction


In [ ]:
bc=next((c for c in ["bulk_log2FC","log2FoldChange"] if c in master),None); sc="median_sn_log2FC" if "median_sn_log2FC" in master else None
if bc and sc:
    d=master[["gene",bc,sc]].copy(); d[bc]=pd.to_numeric(d[bc],errors="coerce"); d[sc]=pd.to_numeric(d[sc],errors="coerce"); d=d.dropna()
    if len(d)>=3:
        fig,ax=plt.subplots(figsize=(6.5,5.5)); ax.scatter(d[bc],d[sc]); ax.axhline(0,linewidth=1); ax.axvline(0,linewidth=1)
        for _,r in d.iterrows():ax.annotate(r.gene,(r[bc],r[sc]),fontsize=8)
        ax.set_xlabel("Bulk SN log2FC (PD vs Control)"); ax.set_ylabel("Median snRNA cell-type log2FC"); ax.set_title("Cross-platform candidate-gene direction")
        fig.tight_layout(); fig.savefig(FIG/"bulk_vs_snRNA_direction.png",dpi=300,bbox_inches="tight"); plt.show()


## 9. Mechanistic prioritization


In [ ]:
if "mechanistic_priority_score" in master:
    d=master.dropna(subset=["mechanistic_priority_score"]).copy(); d["mechanistic_priority_score"]=pd.to_numeric(d.mechanistic_priority_score,errors="coerce"); d=d.sort_values("mechanistic_priority_score")
    fig,ax=plt.subplots(figsize=(8,5.5)); ax.barh(d.gene,d.mechanistic_priority_score); ax.set_xlabel("Mechanistic priority score (not probability)"); ax.set_title("Mechanistic prioritization")
    fig.tight_layout(); fig.savefig(FIG/"mechanistic_priority.png",dpi=300,bbox_inches="tight"); plt.show()


## 10. Exploratory ML — all configurations


In [ ]:
metrics=T["ml_metrics"].copy(); display(metrics); metrics.to_csv(OUT/"ML_metrics_all_models_copy.csv",index=False)
print("Configurations reported:",len(metrics))
print("Notebook 10's Top-5 L2 logistic regression remains the pre-specified primary analysis; do not select the maximum-performing row post hoc as an unbiased winner.")


## 11. Primary LOOCV predictions


In [ ]:
pred=T["ml_predictions"].copy(); display(pred); pred.to_csv(OUT/"primary_LOOCV_predictions_copy.csv",index=False)


## 12. XAI — model explanation, not causality


In [ ]:
xai=G["xai"].copy(); display(xai)
b=detect(xai,["standardized_beta","standardized_logistic_coefficient"],contains=["coefficient"],required=False)
if b:
    d=xai.copy(); d[b]=pd.to_numeric(d[b],errors="coerce"); d=d.dropna(subset=[b]).sort_values(b)
    fig,ax=plt.subplots(figsize=(8,4.8)); ax.barh(d.gene,d[b]); ax.axvline(0,linewidth=1); ax.set_xlabel("Standardized beta"); ax.set_title("Primary model explanation")
    fig.tight_layout(); fig.savefig(FIG/"primary_XAI.png",dpi=300,bbox_inches="tight"); plt.show()


## 13. Thesis-ready compact evidence table


In [ ]:
wanted=["gene","evidence_tier","priority_score","expected_bulk_direction","bulk_log2FC","bulk_padj","median_sn_log2FC","concordance_fraction","n_sn_FDR","n_both_FDR","final_validation_class","cross_dataset_score","mechanistic_priority_score","mechanistic_rank","standardized_beta","dominant_sign_fraction","sign_stable_all_folds","integrated_evidence_category"]
compact=master[[c for c in wanted if c in master]].copy(); compact.to_csv(OUT/"thesis_compact_integrated_evidence_table.csv",index=False); display(compact)


## 14. Quantitative project summary


In [ ]:
s=[{"item":"Primary metabolic signature genes","value":len(master)}]
for cls,n in master.final_validation_class.fillna("Not_available").value_counts().items():s.append({"item":f"Validation class: {cls}","value":int(n)})
if "mechanistic_priority_score" in master:s.append({"item":"Genes with mechanistic evidence","value":int(master.mechanistic_priority_score.notna().sum())})
if "standardized_beta" in master:s.append({"item":"Genes in primary ML/XAI model","value":int(master.standardized_beta.notna().sum())})
summary=pd.DataFrame(s); display(summary); summary.to_csv(OUT/"integrated_project_summary.csv",index=False)


## 15. Integrated interpretation framework

**Discovery:** bulk differential expression and metabolic enrichment identify transcriptomic/pathway signals; pathway recurrence does not make every driver an individually significant DEG.

**Cellular validation:** GSE184950 provides independent dataset/cell-type resolution; donor × cell-type pseudobulk preserves donor-level replication.

**Mechanistic context:** gene–protein–enzyme–reaction links are hypothesis-generating; network centrality is not causal evidence.

**Prediction:** Top-5 ML is exploratory because the small GSE136666 cohort contributed to upstream discovery.

**XAI:** linear log-odds contributions explain fitted model behavior, not biological causality.


## 16. Data-driven Results scaffold


In [ ]:
classes=master.final_validation_class.fillna("Not_available").value_counts().to_dict(); top=ranked.gene.head(min(5,len(ranked))).tolist()
txt=(f"Integrated analysis retained {len(master)} genes from the primary metabolic signature.\n"
     f"Cross-dataset validation classes were distributed as {classes}.\n"
     f"Genes placed earliest in the mechanistic discussion order were {', '.join(top)}.\n"
     "The integrated evidence table combines bulk discovery, single-nucleus validation, biochemical-network context and, for genes in the pre-specified primary classifier, XAI evidence.\n"
     "Prioritization scores were not interpreted as probabilities, and exploratory ML/XAI evidence was not used to upgrade statistical replication status.")
print(txt); (OUT/"results_narrative_scaffold.txt").write_text(txt,encoding="utf-8")


## 17. Final checkpoints


In [ ]:
checks={
"signature_ge_3":len(master)>=3,
"unique_genes":master.gene.is_unique,
"cross_dataset_available":master.final_validation_class.notna().any(),
"mechanistic_available":"mechanistic_priority_score" in master and master.mechanistic_priority_score.notna().any(),
"ML_available":len(T["ml_metrics"])>0,
"XAI_available":"standardized_beta" in master and master.standardized_beta.notna().any(),
"master_exported":(OUT/"integrated_master_gene_evidence.csv").exists(),
"compact_exported":(OUT/"thesis_compact_integrated_evidence_table.csv").exists()}
cp=pd.DataFrame({"check":list(checks),"passed":list(checks.values())}); display(cp)
if not all(checks.values()):raise RuntimeError("Failed checks: "+", ".join(k for k,v in checks.items() if not v))
cp.to_csv(OUT/"notebook12_final_checkpoint.csv",index=False)
manifest={"notebook":"12_integrated_results","primary_signature_n":len(master),"inputs":{k:(str(v) if v else None) for k,v in F.items()},"scores_are_probabilities":False,"ML_independent_validation":False,"causal_claims":False}
with open(OUT/"notebook12_manifest.json","w",encoding="utf-8") as f:json.dump(manifest,f,indent=2,ensure_ascii=False)
print("\n✓ All Notebook 12 checks passed.")
print("Notebook 12 completed successfully.")
print("\n=== FULL ANALYTICAL PIPELINE COMPLETED ===")
